# 16 · Feature Engineering — Columns That Cheat

**Part 2 · Machine Learning Systems** · *Prerequisites: 15* · *Book mapping: DMLS ch. 5*

Features are the interface between the world and the model — and the single most common
place production ML fails. Not because features are weak, but because they **cheat**:
they smuggle in information that won't exist at prediction time, or they exist in two
subtly different versions (training vs serving). Deep models learn features from raw
inputs for images and text; for the tabular data that runs most businesses, engineered
features still rule — so this craft, and its traps, are core curriculum.

**Learning objectives.**

1. Handle missing values by *mechanism* (MCAR/MAR/MNAR) and extract the signal hiding in
   missingness itself.
2. Know when scaling matters, and make a linear model learn an interaction via **feature
   crossing**.
3. Use the **hashing trick** to survive vocabularies that grow after deployment.
4. Inject and then *detect* the three great **leakage** bugs: post-outcome features,
   selection contamination, and temporal leakage.
5. Reproduce **train/serve skew** and fix it the feature-store way: one definition,
   both worlds.

In [ ]:
import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

from sysdes import viz

viz.use_style()
rng = np.random.default_rng(seed=16)

## 1. Missing values have reasons

Three mechanisms, and the treatment depends on which one you face:

- **MCAR** (missing completely at random): a dropped packet. Deletion/simple imputation
  is safe — just costs data.
- **MAR** (missing at random *given other observed columns*): older users skip the
  "phone OS" question. Impute conditionally on what you observe.
- **MNAR** (missing *not* at random — the value itself causes the hiding): high earners
  decline the income question. No imputation recovers the truth — but **the fact of
  missingness is itself a feature.**

In [ ]:
N = 12_000
income = rng.lognormal(mean=10.6, sigma=0.55, size=N)          # true incomes
# MNAR: probability of hiding your income RISES with income.
hidden = rng.random(N) < 1 / (1 + np.exp(-(np.log(income) - 10.9) * 2.2))
reported = np.where(hidden, np.nan, income)

print(f"true mean income: {income.mean():,.0f}, "
      f"mean of REPORTED incomes: {np.nanmean(reported):,.0f}  "
      f"({np.nanmean(reported)/income.mean()-1:+.0%})")
assert np.nanmean(reported) < income.mean() * 0.9, \
    "MNAR: the observed data is biased - deletion or mean-imputation bakes the bias in"

# The signal in the silence: predict 'premium customer' (driven by true income).
is_premium = (income > np.quantile(income, 0.75)).astype(int)
X_imputed = np.nan_to_num(reported, nan=float(np.nanmedian(reported)))[:, None]
X_with_flag = np.c_[X_imputed, hidden.astype(float)]           # + missingness indicator

auc_plain = roc_auc_score(is_premium, LogisticRegression().fit(
    X_imputed, is_premium).predict_proba(X_imputed)[:, 1])
auc_flag = roc_auc_score(is_premium, LogisticRegression().fit(
    X_with_flag, is_premium).predict_proba(X_with_flag)[:, 1])
print(f"AUC imputed-only: {auc_plain:.3f} -> with missingness flag: {auc_flag:.3f}")
assert auc_flag > auc_plain + 0.03, \
    "under MNAR, WHO is missing predicts the target - always ship the indicator column"

Production rules: never let a pipeline silently fill defaults (a `0` that means "unknown"
poisons everything downstream — loud accounting, as in nb 14); impute with train-set
statistics only (a leakage appetizer for §4); and add indicator columns — they're free
and, under MNAR, they're gold.

## 2. Scaling and crossing — two five-minute classics

**Scaling.** Models that use distances (k-NN, SVMs) or gradients (linear, neural) treat a
feature's numeric range as importance. A feature measured in cents will dominate one
measured in years — 100× the range, 100× the vote:

In [ ]:
n = 3_000
signal = rng.normal(0, 1, n)                       # informative, range ~[-3, 3]
y = (signal + rng.normal(0, 0.5, n) > 0).astype(int)
noise_cents = rng.normal(0, 8_000, n)              # uninformative, huge range
X_raw = np.c_[signal, noise_cents]
X_scaled = (X_raw - X_raw[:2000].mean(0)) / X_raw[:2000].std(0)   # TRAIN stats only!

knn_raw = KNeighborsClassifier(15).fit(X_raw[:2000], y[:2000])
knn_scaled = KNeighborsClassifier(15).fit(X_scaled[:2000], y[:2000])
acc_raw = knn_raw.score(X_raw[2000:], y[2000:])
acc_scaled = knn_scaled.score(X_scaled[2000:], y[2000:])
print(f"k-NN accuracy - unscaled: {acc_raw:.3f}, scaled: {acc_scaled:.3f}")
assert acc_raw < 0.6 < 0.8 < acc_scaled, \
    "unscaled: distance is 100% noise-cents; scaled: the signal feature can vote"

(Two habits ride along: compute scaling statistics on the **training split only** — the
test set is the future, and the future doesn't send you its mean — and log-transform
heavy-tailed features like income or latency before scaling, or the tail owns the scale.)

**Crossing.** Linear models can't represent "A *and* B matter together". A feature cross
— the product/conjunction of two features — hands them the interaction as a column:

In [ ]:
# Truth is an interaction: risky IFF (new account AND high amount) or (old AND low).
a, b = rng.normal(0, 1, 4_000), rng.normal(0, 1, 4_000)
y_xor = ((a > 0) ^ (b > 0)).astype(int)

acc_plain = cross_val_score(LogisticRegression(), np.c_[a, b], y_xor, cv=5).mean()
acc_crossed = cross_val_score(LogisticRegression(), np.c_[a, b, a * b], y_xor, cv=5).mean()
print(f"linear model on XOR - raw features: {acc_plain:.3f}, with a×b cross: {acc_crossed:.3f}")
assert acc_plain < 0.6 and acc_crossed > 0.9, \
    "the cross column turned an unlearnable pattern into a linear one"

Trees and deep nets learn interactions themselves (at a data cost); crosses remain the
cheap trick for linear models at scale — with one caveat: crossing categoricals multiplies
cardinalities, which is exactly why the next tool exists.

## 3. The hashing trick — vocabularies that grow

One-hot encoding needs the full category list *at training time*. Production categories
don't cooperate: new cities, new products, new app versions appear the day after deploy.
The **hashing trick** maps categories to a fixed number of buckets via a hash — no
vocabulary to maintain, no unknown-category crash, graceful (measurable) degradation via
collisions:

In [ ]:
def hashed_features(categories: list[str], n_buckets: int) -> np.ndarray:
    X = np.zeros((len(categories), n_buckets))
    for i, cat in enumerate(categories):
        h = int.from_bytes(hashlib.blake2b(cat.encode(), digest_size=8).digest())
        X[i, h % n_buckets] = 1.0
    return X

train_cities = [f"city_{i}" for i in range(400)]
new_city = "city_9999"                                  # born after deployment

# One-hot with a frozen vocabulary: the unseen category has NO column.
vocab = {c: i for i, c in enumerate(train_cities)}
assert new_city not in vocab, "one-hot must either crash or emit an all-zero row"
# Hashed: any string - past, present, or future - gets a bucket. No crash, ever.
assert hashed_features([new_city], 256).sum() == 1.0
print("unseen category: one-hot has no column for it; hashing just... hashes it")

# The price is collisions. Measure the fraction of categories sharing a bucket:
def collision_rate(cats, n_buckets):
    buckets = hashed_features(cats, n_buckets).argmax(1)
    counts = np.bincount(buckets, minlength=n_buckets)
    return float(np.mean([counts[b] > 1 for b in buckets]))

dims = [64, 128, 256, 512, 1024, 4096]
rates = [collision_rate(train_cities, d) for d in dims]
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(dims, rates, "o-")
ax.set_xscale("log")
ax.set_xlabel("hash buckets")
ax.set_ylabel("share of categories colliding")
ax.set_title("400 categories: collisions vs table size (birthday arithmetic)")
plt.show()
assert rates[0] > 0.9 and rates[-1] < 0.1, "size the table ~10x the live vocabulary"

Collisions blur categories together (two cities share a weight) — usually a tiny,
graceful tax, and models often *shrug* at it. The trick powers ad-tech systems with
billions of sparse features; exercise 3 does the sizing math.

## 4. Leakage — the silent killer

> **Data leakage**: information available in training features that will **not** be
> available (or true) at prediction time. The model aces every offline evaluation and
> fails in production — the worst possible failure order, because you find out *after*
> shipping.

Three classic strains. We inject each, watch the fraudulent score, then detect it.

### 4.1 Post-outcome features (target leakage)

A fraud model's training table includes `refund_issued` — a column set by the fraud team
*after confirming fraud*. It is the label wearing a trench coat:

In [ ]:
n = 8_000
amount = rng.exponential(80, n)
is_new_account = (rng.random(n) < 0.3).astype(float)
fraud = ((0.008 * amount + 1.1 * is_new_account + rng.normal(0, 1, n)) > 2.2).astype(int)
refund_issued = (fraud & (rng.random(n) < 0.92)).astype(float)   # written AFTER the fact

X_leaky = np.c_[amount, is_new_account, refund_issued]
X_honest = np.c_[amount, is_new_account]
auc_leaky = cross_val_score(LogisticRegression(), X_leaky, fraud,
                            cv=5, scoring="roc_auc").mean()
auc_honest = cross_val_score(LogisticRegression(), X_honest, fraud,
                             cv=5, scoring="roc_auc").mean()
print(f"offline AUC - with refund_issued: {auc_leaky:.3f}, honest features: {auc_honest:.3f}")
assert auc_leaky > 0.95 > auc_honest, "the leak buys ~perfection offline, zero online"

# DETECTION: screen every feature alone. A single feature that nearly matches
# the model's whole AUC is either a great feature or a time traveler - audit it.
for i, name in enumerate(["amount", "is_new_account", "refund_issued"]):
    solo = roc_auc_score(fraud, X_leaky[:, i])
    flag = "  <-- suspicious" if solo > 0.9 else ""
    print(f"  single-feature AUC {name:15s}: {max(solo, 1-solo):.3f}{flag}")
assert roc_auc_score(fraud, refund_issued) > 0.9

The detector is a *process*, not just a scan: for every feature ask **"at the moment of
prediction, is this value already written — and would it have this value if the outcome
hadn't happened yet?"** Post-outcome columns enter tables through joins with
operational systems (tickets, refunds, shipping status); schema timestamps (nb 14) are
your witnesses.

### 4.2 Selection contamination (the 2,000-noise-features scandal)

Preprocessing fitted on **all** data before splitting leaks the test set into training.
The most dramatic case — feature selection — can conjure skill out of pure noise:

In [ ]:
n_samples, n_features = 100, 2_000
X_noise = rng.normal(0, 1, (n_samples, n_features))
y_coin = (rng.random(n_samples) < 0.5).astype(int)      # PURE COIN FLIPS

# WRONG: choose the 20 features most correlated with y on the FULL dataset,
# then cross-validate on those. The selection saw the test folds' labels.
corr = np.abs([np.corrcoef(X_noise[:, j], y_coin)[0, 1] for j in range(n_features)])
top = np.argsort(-corr)[:20]
score_wrong = cross_val_score(LogisticRegression(), X_noise[:, top], y_coin, cv=5).mean()

# RIGHT: selection happens INSIDE each fold, on training data only.
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())
score_right = cross_val_score(pipe, X_noise, y_coin, cv=5).mean()

print(f"predicting COIN FLIPS from NOISE - contaminated CV: {score_wrong:.2f}, "
      f"clean pipeline CV: {score_right:.2f}")
assert score_wrong > 0.65, "selection leakage manufactured skill from nothing"
assert abs(score_right - 0.5) < 0.12, "done right, noise scores like noise"

Read that again: **65–75% cross-validated accuracy predicting coin flips.** With 2,000
noise columns, ~dozens correlate with any labels by luck; selecting on the full data
picks exactly those, and CV then "confirms" them. The rule that prevents the whole class
of bug: *every* fitted step — selection, scaling, imputation, target encoding — lives
inside the pipeline, fitted per training fold. (sklearn's `Pipeline` exists precisely to
make the wrong thing hard.)

### 4.3 Temporal leakage — the random split that peeks

When the world drifts (nb 19), a random split scatters the future into your training set.
The honest split for a model that will serve *the future* is a **time split**:

In [ ]:
n = 6_000
t = np.linspace(0, 1, n)                                # time
x1, x2 = rng.normal(0, 1, n), rng.normal(0, 1, n)
# The true relationship ROTATES over time (concept drift):
w1, w2 = np.cos(2.2 * t), np.sin(2.2 * t)
y_drift = ((w1 * x1 + w2 * x2 + rng.normal(0, 0.4, n)) > 0).astype(int)
X_drift = np.c_[x1, x2]

shuffled = rng.permutation(n)
rand_train, rand_test = shuffled[:4200], shuffled[4200:]
score_random = LogisticRegression().fit(X_drift[rand_train], y_drift[rand_train]) \
    .score(X_drift[rand_test], y_drift[rand_test])
score_time = LogisticRegression().fit(X_drift[:4200], y_drift[:4200]) \
    .score(X_drift[4200:], y_drift[4200:])

print(f"random split: {score_random:.3f} vs time split: {score_time:.3f}")
assert score_random > score_time + 0.05, \
    "the random split trained on the future and inflated the score; production gets the time split"

The time split's lower number is not worse — it is **the truth about deployment**, where
training data is always the past. Same family: *group leakage* (one user's rows on both
sides of the split — the model memorizes users, not patterns); split by user, by session,
by hospital — by whatever unit must be unseen.

## 5. Train/serve skew — one feature, two truths

The subtlest production failure: the feature is computed by **two implementations** — a
batch pipeline for training, a service for serving — and they disagree. The model is
fine; its *inputs* changed dialect. Canonical bug: "user's average purchase amount",
where the training SQL includes the current transaction in the average and the serving
code (correctly!) uses only prior history:

In [ ]:
events = pd.DataFrame({
    "user_id": rng.integers(0, 300, 5_000),
    "amount": np.round(rng.exponential(60, 5_000), 2),
}).sort_values("user_id", kind="stable").reset_index(drop=True)

# TRAINING pipeline (buggy): groupby-mean includes the current row - each row
# gets a peek at its own value. Offline, features look great...
train_feature = events.groupby("user_id")["amount"].transform("mean")

# SERVING pipeline (honest): only PRIOR events exist at prediction time.
def serving_feature(df):
    prior_mean = (df.groupby("user_id")["amount"]
                    .apply(lambda s: s.shift(1).expanding().mean())
                    .reset_index(level=0, drop=True))
    return prior_mean.fillna(df["amount"].mean())      # cold start: global mean

serve_feature = serving_feature(events)
mismatch = float(np.mean(np.abs(train_feature - serve_feature) > 0.01))
print(f"rows where training and serving disagree on the SAME feature: {mismatch:.0%}")
assert mismatch > 0.9, "TRAIN/SERVE SKEW: the model was trained on a feature that "\
    "does not exist in production"

# THE FIX: one definition, used by both worlds. Training rebuilds its table by
# running the SERVING definition over history (point-in-time correctness).
train_feature_fixed = serving_feature(events)
assert (train_feature_fixed == serve_feature).all(), \
    "one shared transformation -> zero skew, by construction"
print("fixed: training features = replaying the serving definition over history")

Generalize the fix and you've invented the **feature store**: a system that keeps *one
versioned definition* per feature and materializes it twice — into an **offline store**
(historical, point-in-time-correct values for training; the date-partitioned lake of
nb 14) and an **online store** (latest values at millisecond latency for serving; the
key-value layout of nb 03/06 — nb 14 exercise 2's answer). Same definition, two layouts,
derived from one event log: it is nb 12's unbundled-database pattern with a payroll. The
point-in-time discipline ("a training row may only see features as they were at that
row's timestamp") returns as the heart of capstone 25.

## 6. Exercises

1. **Mechanism triage.** Classify and treat: (a) survey income skipped by high earners;
   (b) sensor readings lost when a batch job crashes; (c) "years at employer" blank for
   self-employed applicants; (d) heart-rate missing for patients too sick to wear the
   band. Which ones deserve indicator features most urgently?
2. **Target encoding.** Mean-encoding (`city → mean(y) in that city`) computed over the
   full training set leaks each row's own label into its feature. Show why rare
   categories make it catastrophic, and describe the out-of-fold (or leave-one-out +
   smoothing) fix — and where the *serving* value comes from.
3. **Hash sizing.** With V live categories and m buckets, P(a given category collides) ≈
   1 − e^(−V/m). Size m for 5,000 categories at <2% collision, and check the §3 plot's
   agreement. Why do collisions hurt less for *frequent* categories?
4. **Leakage audit.** A loan-default model's features: requested_amount, credit_score,
   days_since_application, collections_agency_assigned, application_channel,
   account_closed_reason, avg_balance_last_90d. Name the leaks and the tell-tale sign
   each would show in §4.1's single-feature screen.
5. **Feature store triage.** For a ride-price model, place each feature: city_base_rate
   (changes weekly), driver_rating (changes per trip), current_surge (changes per
   second), rider_signup_date (never changes). Which store serves each (offline-only /
   online materialized / request-time computed), and which needs streaming (nb 11)?

### Solutions

**1.** (a) MNAR → indicator feature mandatory + model the bias (or collect differently).
(b) MCAR → impute/drop freely; fix the job (nb 14's audit would have caught it).
(c) MAR-ish *structural* missingness (not employed → field undefined) → indicator +
imputation conditional on employment type; arguably a separate feature, not a missing
one. (d) MNAR and *dangerous*: missingness correlates with the outcome — indicator
essential, and clinical review before shipping (the model may learn "no data = dying").

**2.** A city with 2 rows gets an encoding that is ~half its own label — memorization
served as a feature; rare categories are label lookup tables. Fix: compute each row's
encoding from *other* folds (out-of-fold), plus smoothing toward the global mean by
category count. Serving uses the encoding table computed on **all** training data (no
row's own label exists at serving) — note the asymmetry: the leak is a *training-time*
phenomenon.

**3.** m ≈ −V / ln(0.98) ≈ 5,000/0.0202 ≈ ~250k buckets (or accept 2 hash functions /
larger dims). Frequent categories dominate their bucket's weight during training — the
collision partner is statistical noise to them; rare categories can be drowned by a
frequent collider (mirror of exercise 2's rare-category fragility).

**4.** `collections_agency_assigned` (assigned after default — §4.1's refund clone) and
`account_closed_reason` (written at account death) are post-outcome; both would show
absurd solo AUC. `days_since_application` is subtler: if the table was snapshotted
*after* outcomes, it encodes outcome timing (defaulted loans stop aging) — a timestamp
audit catches it. The rest are legitimate if computed point-in-time
(`avg_balance_last_90d` must be as-of application date — §5's replay).

**5.** rider_signup_date: offline + copied online (static). city_base_rate: offline
table refreshed weekly, cached online. driver_rating: online store updated by a
per-trip stream consumer (nb 11). current_surge: too fresh for any store — computed
request-time from a live aggregate (a streaming windowed count, nb 11 §3, exposed as a
service). The split *is* the feature store design.

## 7. Takeaways

- Missingness has mechanisms; MNAR means the silence is signal — ship indicator columns
  and impute with train statistics only.
- Scale for distance/gradient models; cross for linear ones; hash for vocabularies that
  refuse to stand still (size ~10× the vocabulary).
- Leakage = training on information the future won't have. The three strains: post-
  outcome columns (audit: "was this written yet?"), full-data preprocessing (cure: fit
  everything inside the CV pipeline), and random splits over drifting time (cure: split
  like production — by time, by user, by group).
- A *lower* honest score beats a higher fraudulent one — the honest number is the only
  one production will ever pay.
- Train/serve skew: one feature, one definition, both worlds — that principle, plus
  point-in-time history, is the feature store.

**Further reading.** DMLS ch. 5; Kaufman et al., *Leakage in Data Mining* (2012);
Hastie et al., *Elements of Statistical Learning* §7.10.2 (the wrong-way CV story).

**Next:** [17 · Model development & evaluation](17-model-dev-evaluation.ipynb) — with
honest features in hand: baselines, experiment tracking, and evaluations that find the
failures averages hide.